# Google Play Store Intelligence Dashboard

# Feature Engineering

## Overview

Feature engineering is the process of creating new variables from existing data to improve analysis and generate meaningful business insights.

In this notebook, we will create business-friendly features that will later be used in exploratory data analysis, dashboards, and business recommendations.

---

# Objectives

The objectives of this notebook are:

- Convert app size into a consistent numeric format
- Categorize installs into business-friendly buckets
- Categorize app ratings
- Categorize app prices
- Categorize review counts
- Measure app freshness
- Prepare a feature-rich dataset for analysis

#  Import Libraries

In [48]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

# Load Clean Dataset

In [49]:
apps_clean = pd.read_csv("../data/processed/cleaned_googleplaystore.csv")
reviews_clean = pd.read_csv("../data/processed/cleaned_googleplaystore_user_reviews.csv")

print(apps_clean.shape)

(8892, 13)


#  Preview Dataset

In [50]:
apps_clean.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,10000,Free,0.0,Everyone,Art & Design,2018-01-07,1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,2018-01-15,2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,5000000,Free,0.0,Everyone,Art & Design,2018-08-01,1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,50000000,Free,0.0,Teen,Art & Design,2018-06-08,Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,100000,Free,0.0,Everyone,Art & Design;Creativity,2018-06-20,1.1,4.4 and up


# Feature 1 — Convert App Size

The **Size** column contains values such as:

- 19M
- 500k
- Varies with device

To simplify analysis, all sizes will be converted into **megabytes (MB)**.

In [51]:
def convert_size(size):

    if pd.isna(size):
        return np.nan

    size = str(size)

    if size == "Varies with device":
        return np.nan

    if size.endswith("M"):
        return float(size[:-1])

    if size.endswith("k"):
        return float(size[:-1]) / 1024

    return np.nan


apps_clean["Size_MB"] = apps_clean["Size"].apply(convert_size)

In [52]:
apps_clean[["Size", "Size_MB"]].head(10)

,Size,Size_MB
0,19M,19.0
1,14M,14.0
2,8.7M,8.7
3,25M,25.0
4,2.8M,2.8
5,5.6M,5.6
6,19M,19.0
7,29M,29.0
8,33M,33.0
9,3.1M,3.1


# Feature 2 — Install Bucket

Install counts vary from a few hundred downloads to over one billion.

Grouping install counts into buckets makes trends easier to analyze.

In [53]:
def install_bucket(installs):

    if installs < 10000:
        return "Low"

    elif installs < 100000:
        return "Medium"

    elif installs < 1000000:
        return "High"

    elif installs < 10000000:
        return "Very High"

    else:
        return "Viral"


apps_clean["Install_Bucket"] = apps_clean["Installs"].apply(install_bucket)

In [54]:
apps_clean["Install_Bucket"].value_counts()

Install_Bucket
Very High    2169
Viral        1883
Low          1763
High         1626
Medium       1451
Name: count, dtype: int64

# Feature 3 — Rating Group

Ratings are grouped into performance categories to simplify business analysis.

In [55]:
def rating_group(rating):

    if pd.isna(rating):
        return "No Rating"

    elif rating < 3:
        return "Poor"

    elif rating < 4:
        return "Average"

    elif rating < 4.5:
        return "Good"

    else:
        return "Excellent"


apps_clean["Rating_Group"] = apps_clean["Rating"].apply(rating_group)

In [56]:
apps_clean["Rating_Group"].value_counts()

Rating_Group
Good         4132
Excellent    2815
Average      1665
Poor          280
Name: count, dtype: int64

#  Feature 4 — Price Category

Apps are categorized based on pricing strategy.

In [57]:
def price_category(price):

    if price == 0:
        return "Free"

    elif price <= 2:
        return "Budget"

    elif price <= 10:
        return "Premium"

    else:
        return "Luxury"


apps_clean["Price_Category"] = apps_clean["Price"].apply(price_category)

In [58]:
apps_clean["Price_Category"].value_counts()

Price_Category
Free       8279
Premium     348
Budget      208
Luxury       57
Name: count, dtype: int64

# Feature 5 — Review Bucket

Applications are grouped according to the number of user reviews received.

In [59]:
def review_bucket(review):

    if review < 100:
        return "Very Low"

    elif review < 1000:
        return "Low"

    elif review < 10000:
        return "Medium"

    elif review < 100000:
        return "High"

    else:
        return "Very High"


apps_clean["Review_Bucket"] = apps_clean["Reviews"].apply(review_bucket)

In [60]:
apps_clean["Review_Bucket"].value_counts()

Review_Bucket
Very High    1971
High         1958
Very Low     1899
Medium       1556
Low          1508
Name: count, dtype: int64

# Feature 6 — Years Since Last Update

This feature measures how recently an application has been updated.

In [61]:
apps_clean["Last Updated"] = pd.to_datetime(
    apps_clean["Last Updated"]
)

reference_date = pd.Timestamp("2018-08-31")

apps_clean["Years_Since_Update"] = (
    reference_date -
    apps_clean["Last Updated"]
).dt.days / 365

In [62]:
apps_clean[
    ["Last Updated", "Years_Since_Update"]
].head()

,Last Updated,Years_Since_Update
0,2018-01-07,0.646575
1,2018-01-15,0.624658
2,2018-08-01,0.082192
3,2018-06-08,0.230137
4,2018-06-20,0.197260


# Feature 7 — Update Status

Applications are grouped according to how recently they have been updated.

In [63]:
def update_status(years):

    if years < 1:
        return "Recently Updated"

    elif years < 2:
        return "Moderately Updated"

    else:
        return "Outdated"


apps_clean["Update_Status"] = (
    apps_clean["Years_Since_Update"]
    .apply(update_status)
)

In [64]:
apps_clean["Update_Status"].value_counts()

Update_Status
Recently Updated      6755
Outdated              1070
Moderately Updated    1067
Name: count, dtype: int64

# Bonus Feature 8 — Market Leader

Applications with over **10 million installs** are considered market leaders.

In [65]:
apps_clean["Market_Leader"] = np.where(
    apps_clean["Installs"] >= 10000000,
    "Yes",
    "No"
)

In [66]:
apps_clean["Market_Leader"].value_counts()

Market_Leader
No     7009
Yes    1883
Name: count, dtype: int64

# Bonus Feature 9 — Engagement Ratio

The engagement ratio estimates how actively users review an application relative to the number of installs.

Formula:

Engagement Ratio = Reviews / Installs

In [67]:
apps_clean["Engagement_Ratio"] = (
    apps_clean["Reviews"] /
    apps_clean["Installs"]
)

In [68]:
apps_clean[
    ["App", "Reviews", "Installs", "Engagement_Ratio"]
].head()

,App,Reviews,Installs,Engagement_Ratio
0,Photo Editor & Candy Camera & Grid & ScrapBook,159,10000,0.015900
1,Coloring book moana,967,500000,0.001934
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",87510,5000000,0.017502
3,Sketch - Draw & Paint,215644,50000000,0.004313
4,Pixel Draw - Number Art Coloring Book,967,100000,0.009670


# Verify Engineered Features

In [69]:
apps_clean.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Size_MB,Install_Bucket,Rating_Group,Price_Category,Review_Bucket,Years_Since_Update,Update_Status,Market_Leader,Engagement_Ratio
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,10000,Free,0.0,Everyone,Art & Design,2018-01-07,1.0.0,4.0.3 and up,19.0,Medium,Good,Free,Low,0.646575,Recently Updated,No,0.015900
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,2018-01-15,2.0.0,4.0.3 and up,14.0,High,Average,Free,Low,0.624658,Recently Updated,No,0.001934
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,5000000,Free,0.0,Everyone,Art & Design,2018-08-01,1.2.4,4.0.3 and up,8.7,Very High,Excellent,Free,High,0.082192,Recently Updated,No,0.017502
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,50000000,Free,0.0,Teen,Art & Design,2018-06-08,Varies with device,4.2 and up,25.0,Viral,Excellent,Free,Very High,0.230137,Recently Updated,Yes,0.004313
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,100000,Free,0.0,Everyone,Art & Design;Creativity,2018-06-20,1.1,4.4 and up,2.8,High,Good,Free,Low,0.197260,Recently Updated,No,0.009670


In [70]:
apps_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 8892 entries, 0 to 8891
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   App                 8892 non-null   str           
 1   Category            8892 non-null   str           
 2   Rating              8892 non-null   float64       
 3   Reviews             8892 non-null   int64         
 4   Size                8892 non-null   str           
 5   Installs            8892 non-null   int64         
 6   Type                8892 non-null   str           
 7   Price               8892 non-null   float64       
 8   Content Rating      8892 non-null   str           
 9   Genres              8892 non-null   str           
 10  Last Updated        8892 non-null   datetime64[us]
 11  Current Ver         8888 non-null   str           
 12  Android Ver         8890 non-null   str           
 13  Size_MB             7424 non-null   float64       
 14  Ins

#  Save Feature Engineered Dataset

In [71]:
apps_clean.to_csv(
    "../data/processed/featured_googleplaystore.csv",
    index=False
)

# Feature Engineering Summary

The following business-focused features were successfully created:

- ✅ Size_MB
- ✅ Install_Bucket
- ✅ Rating_Group
- ✅ Price_Category
- ✅ Review_Bucket
- ✅ Years_Since_Update
- ✅ Update_Status
- ✅ Market_Leader
- ✅ Engagement_Ratio

These features will simplify analysis and help generate meaningful business insights in the next stage of the project.

In [72]:
apps_clean[
    [
        "App",
        "Category",
        "Rating",
        "Size_MB",
        "Install_Bucket",
        "Rating_Group",
        "Price_Category",
        "Review_Bucket",
        "Years_Since_Update",
        "Update_Status",
        "Market_Leader",
        "Engagement_Ratio",
    ]
].head(10)

,App,Category,Rating,Size_MB,Install_Bucket,Rating_Group,Price_Category,Review_Bucket,Years_Since_Update,Update_Status,Market_Leader,Engagement_Ratio
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,19.0,Medium,Good,Free,Low,0.646575,Recently Updated,No,0.015900
1,Coloring book moana,ART_AND_DESIGN,3.9,14.0,High,Average,Free,Low,0.624658,Recently Updated,No,0.001934
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,8.7,Very High,Excellent,Free,High,0.082192,Recently Updated,No,0.017502
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,25.0,Viral,Excellent,Free,Very High,0.230137,Recently Updated,Yes,0.004313
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,2.8,High,Good,Free,Low,0.197260,Recently Updated,No,0.009670
5,Paper flowers instructions,ART_AND_DESIGN,4.4,5.6,Medium,Good,Free,Low,1.432877,Moderately Updated,No,0.003340
6,Smoke Effect Photo Maker - Smoke Editor,ART_AND_DESIGN,3.8,19.0,Medium,Average,Free,Low,0.347945,Recently Updated,No,0.003560
7,Infinite Painter,ART_AND_DESIGN,4.1,29.0,Very High,Good,Free,High,0.213699,Recently Updated,No,0.036815
8,Garden Coloring Book,ART_AND_DESIGN,4.4,33.0,Very High,Good,Free,High,0.945205,Recently Updated,No,0.013791
9,Kids Paint Free - Drawing Fun,ART_AND_DESIGN,4.7,3.1,Medium,Excellent,Free,Low,0.161644,Recently Updated,No,0.012100


# Next Step

Proceed to:

## 04_Exploratory_Data_Analysis.ipynb

In the next notebook, we will answer key business questions such as:

- Which app categories dominate the Play Store?
- Which categories receive the highest installs?
- Do paid apps perform better than free apps?
- Does app size influence popularity?
- Do recently updated apps receive higher ratings?
- Which categories present the best growth opportunities?